# LLM Benchmarking + Fine-Tuning for Financial Sentiment (SEntFiN-v1.1)

**Question this notebook answers**: FinBERT+LoRA hit 86.5% F1 on SEntFiN, but has a confirmed,
confidently-wrong blind spot on "corporate action" headlines (partnerships, new labs,
contract wins) that don't contain classic profit/loss vocabulary. Which open LLM -- across
several different model families, not just one -- actually does best here, zero-shot and
fine-tuned, and does any of them fix that blind spot?

**Two-stage design, so compute goes where it's actually earned**:

**Stage 1 -- broad zero-shot benchmark (cheap, no training)**: 9 candidates spanning ~0.5B-7B
params across 5 different organizations/families -- Qwen2.5 (0.5B/1.5B/3B/7B-Instruct),
Gemma-2-2B-it, Phi-3.5-mini-instruct, SmolLM2-1.7B-Instruct, and Llama-3.2-1B/3B-Instruct
(gated on HF; skipped gracefully if you haven't accepted the license / set `HF_TOKEN`). Each
gets scored zero-shot via log-probability extraction (the CLaC@FinMMEval-2026 paper's method:
one forward pass, softmax restricted to the positive/negative/neutral token IDs -- no text
generation, no fine-tuning at all). This alone tells you whether broad pretraining already
beats a narrow fine-tuned model on FinBERT's failure genre, before spending anything on training.

**Stage 2 -- LoRA + QLoRA fine-tuning (expensive, so only the `TOP_K_FOR_FINETUNING` zero-shot
winners advance)**: fine-tuning all 9 candidates with a full 25-trial search each would be
~4.5x the compute of fine-tuning just the top 2 -- this is the standard way to keep a
multi-model benchmark's compute bounded without giving up real search rigor where it counts.
LoRA uses bf16 base weights + trainable low-rank adapters; QLoRA quantizes the frozen base to
4-bit NF4 first (smaller footprint, useful as models get bigger), both applied via
`AutoModelForSequenceClassification` on the decoder backbone (analogous to what FinBERT+LoRA
did on an encoder). Whichever models "win" are also all real Ollama models
(`ollama pull qwen2.5:1.5b`, `gemma2:2b`, `phi3.5`, etc.) for local serving afterward.

Each fine-tuning method gets its own Optuna hyperparameter search (rank/alpha/dropout/lr/
weight_decay/batch_size/target_modules -- see the dedicated section below) before committing
to a full training run, mirroring how Part A tuned its classical/DL models and Part B tuned
its RL agents — "max achievable" means actually searching, not guessing once.

**Requires a GPU.** Runs 4-bit quantization (bitsandbytes) and up to 7B-parameter inference /
3B-parameter fine-tuning — infeasible on an 8GB-RAM CPU-only machine. Run this on Kaggle (free
T4, Settings -> Accelerator -> GPU T4 x2) or any other CUDA GPU.


In [20]:
# ══════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install("transformers>=4.46", "peft>=0.13", "bitsandbytes>=0.44", "accelerate>=1.0",
            "datasets", "scikit-learn", "mlflow", "sentencepiece", "optuna", "optuna-integration",
            "torchao>=0.16.0")
# Kaggle's base image ships an older torchao (0.10.0) than current peft requires (>=0.16.0).
# peft's LoRA-layer dispatch checks torchao's availability unconditionally on EVERY adapter
# attachment (regardless of whether the model actually uses torchao quantization), and that
# check raises ImportError -- not a clean "unavailable" -- when an incompatible version is
# merely installed. Upgrading it here avoids get_peft_model() crashing on every model.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.0 MB/s eta 0:00:00


In [21]:
import os
import ast
import gc
import json
import logging

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import mlflow
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


In [22]:
# ══════════════════════════════════════════════════════════════
# CONFIG — split/seed identical to notebooks/07_finbert_peft.ipynb for a fair comparison
# ══════════════════════════════════════════════════════════════
LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {0: "positive", 1: "neutral", 2: "negative"}
NUM_LABELS = 3

SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.10
MAX_LENGTH = 128

# Stage 1 casts a wide net across DIFFERENT model families and sizes (~0.5B-7B) -- zero-shot
# inference is cheap (no training), so there's no reason to guess which family is best up
# front. Stage 2 (the expensive LoRA/QLoRA search) only runs on whichever TOP_K win this
# screen -- fine-tuning all 9 with a 25-trial search each would be ~4.5x the compute for a
# question this cheap screen already answers well enough to narrow down.
# "quantize": True loads that model 4-bit (NF4) even for zero-shot inference -- a T4 only has
# ~14.5GB usable, and a 7B model's weights ALONE cost ~14GB in bf16 (7B x 2 bytes), leaving
# essentially no headroom for activations. This exact model OOM'd during the forward pass
# (not loading) on a first pass at this notebook; 4-bit cuts that to ~4GB, leaving real margin.
ZERO_SHOT_CANDIDATES = {
    "Qwen2.5-0.5B-Instruct": {"hf_id": "Qwen/Qwen2.5-0.5B-Instruct", "quantize": False},
    "Qwen2.5-1.5B-Instruct": {"hf_id": "Qwen/Qwen2.5-1.5B-Instruct", "quantize": False},
    "Qwen2.5-3B-Instruct": {"hf_id": "Qwen/Qwen2.5-3B-Instruct", "quantize": False},
    "Qwen2.5-7B-Instruct": {"hf_id": "Qwen/Qwen2.5-7B-Instruct", "quantize": True},
    "Gemma-2-2B-it": {"hf_id": "google/gemma-2-2b-it", "quantize": False},
    "Phi-3.5-mini-instruct": {"hf_id": "microsoft/Phi-3.5-mini-instruct", "quantize": False},   # 3.8B, still fits bf16
    "SmolLM2-1.7B-Instruct": {"hf_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "quantize": False},
    # Meta gates these on HF (needs license acceptance + HF_TOKEN) -- included best-effort;
    # the zero-shot loop below skips any candidate that fails to load rather than crashing.
    "Llama-3.2-1B-Instruct": {"hf_id": "meta-llama/Llama-3.2-1B-Instruct", "quantize": False},
    "Llama-3.2-3B-Instruct": {"hf_id": "meta-llama/Llama-3.2-3B-Instruct", "quantize": False},
}
TOP_K_FOR_FINETUNING = 2   # only this many advance from the zero-shot screen to the expensive stage

# A REAL search: 25 trials per (model, method) combo, so TOP_K=2 models x {LoRA, QLoRA} = 4
# combos x 25 trials = 100 trials total.
# using TPE sampling (learns from prior trials instead of guessing blind) plus median pruning
# (kills a trial the moment it's tracking worse than the median of prior trials at the same
# epoch, so most of the 100 trials never run to completion -- this is what makes 25 trials
# actually affordable instead of 25x the compute of one). Adjust SEARCH_TRIALS down only if
# your Kaggle GPU-hour quota is tight; pruning already does most of the budget protection.
SEARCH_TRIALS = 25
SEARCH_EPOCHS = 4       # ceiling a trial can run to before pruning judges it "converged enough"
FULL_EPOCHS = 10
PATIENCE = 3

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. QLoRA (4-bit) requires CUDA -- this notebook will fail "
          "or be impractically slow without one. On Kaggle: Settings -> Accelerator -> GPU T4 x2.")

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

mlflow.set_tracking_uri("sqlite:///" + os.path.join(BASE_DIR, "mlflow.db"))
mlflow.set_experiment("IndianFinSentiment")   # same experiment as FinBERT+LoRA -- directly comparable


Using device: cuda


<Experiment: artifact_location='/kaggle/working/mlruns/1', creation_time=1785582851894, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785582851894, lifecycle_stage='active', name='IndianFinSentiment', tags={}, trace_location=None, workspace='default'>

## Data — SEntFiN-v1.1, identical split to the FinBERT+LoRA notebook

Same loading rule: keep headlines where every mentioned entity shares one sentiment label
(single-label headlines, plus the ~88.5% of multi-entity headlines that happen to agree),
drop the ~11.5% where entities disagree (ambiguous for a single whole-headline label).
Same 70/15/15 stratified split, same seed (42) -- so any accuracy difference from FinBERT+LoRA
is attributable to the model/method, not a different train/test split.


In [23]:
def resolve_sentifin_path() -> str:
    candidates = ["../data/raw/SEntFiN-v1.1.csv", "../data/SEntFiN-v1.1.csv"]
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f == "SEntFiN-v1.1.csv":
                    candidates.insert(0, os.path.join(root, f))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "SEntFiN-v1.1.csv not found. On Kaggle: Add Data -> upload SEntFiN-v1.1.csv as a "
        "dataset and attach it to this notebook. Locally: place it at data/raw/SEntFiN-v1.1.csv."
    )


def load_sentifin() -> pd.DataFrame:
    # NOTE: uses ast.literal_eval, not json.loads(text.replace("'", '"')) -- the Decisions
    # column is often already valid double-quoted JSON, and a handful of rows contain an
    # apostrophe INSIDE a value (e.g. "Moody's", "Dr Reddy's Labs"). A blind quote-replace
    # corrupts exactly those rows into invalid JSON and crashes the whole load. literal_eval
    # (wrapped in try/except, matching notebooks/07_finbert_peft.ipynb's proven loader) parses
    # the actual Python-literal structure regardless of which quote style is used.
    log.info("Loading SEntFiN-v1.1...")
    raw = pd.read_csv(resolve_sentifin_path())

    texts, labels = [], []
    dropped_unparseable, dropped_ambiguous = 0, 0
    for _, row in raw.iterrows():
        try:
            decisions = ast.literal_eval(row["Decisions"])
        except (ValueError, SyntaxError):
            dropped_unparseable += 1
            continue
        if not decisions:
            continue
        unique_labels = set(decisions.values())
        if len(unique_labels) != 1:
            dropped_ambiguous += 1
            continue
        texts.append(row["Title"])
        labels.append(unique_labels.pop())

    df = pd.DataFrame({"text": texts, "label": labels})
    df["label"] = df["label"].str.lower().str.strip()
    df = df[df["label"].isin(LABEL2ID.keys())]
    df["label_id"] = df["label"].map(LABEL2ID)
    log.info(f"Loaded {len(df)} single-label headlines "
             f"(dropped {dropped_ambiguous} ambiguous, {dropped_unparseable} unparseable)")
    log.info(f"Label distribution: {df['label'].value_counts().to_dict()}")
    return df


df = load_sentifin()

train_val_df, test_df = train_test_split(df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=SEED)
train_df, val_df = train_test_split(
    train_val_df, test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=train_val_df["label_id"], random_state=SEED
)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")


2026-08-01 13:04:37,279 | INFO | Loading SEntFiN-v1.1...
2026-08-01 13:04:37,970 | INFO | Loaded 9518 single-label headlines (dropped 1235 ambiguous, 0 unparseable)
2026-08-01 13:04:37,972 | INFO | Label distribution: {'neutral': 3443, 'positive': 3364, 'negative': 2711}


train=7138 val=952 test=1428


## Stage 1 — Zero-shot benchmark across 9 models, log-probability extraction (no fine-tuning at all)

Following the CLaC@FinMMEval-2026 paper's approach: one forward pass through the frozen
instruct model, extract the logits at the final token position, restrict the softmax to the
token-ID variants for {positive, negative, neutral} (both with/without a leading space, both
cases -- avoids brittleness to a single tokenization), and take the argmax. No text generation,
no fine-tuning -- purely testing what the model already knows from pretraining.


In [25]:
ZERO_SHOT_PROMPT = (
    "You are a financial sentiment classifier for Indian stock market headlines. "
    "Classify the sentiment of the following headline as exactly one word: "
    "positive, negative, or neutral.\n\nHeadline: {headline}\nSentiment:"
)


def get_label_token_ids(tokenizer):
    """All BPE variants of each label word (leading-space / no-space, lower/title case)."""
    label_variants = {
        "positive": ["positive", " positive", "Positive", " Positive"],
        "negative": ["negative", " negative", "Negative", " Negative"],
        "neutral": ["neutral", " neutral", "Neutral", " Neutral"],
    }
    label_token_ids = {}
    for label, variants in label_variants.items():
        ids = set()
        for v in variants:
            toks = tokenizer.encode(v, add_special_tokens=False)
            if len(toks) >= 1:
                ids.add(toks[0])   # first token of the label word
        label_token_ids[label] = list(ids)
    return label_token_ids


@torch.no_grad()
def zero_shot_predict_batch(model, tokenizer, texts, label_token_ids, batch_size=16):
    model.eval()
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        prompts = [ZERO_SHOT_PROMPT.format(headline=t) for t in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(model.device)
        logits = model(**enc).logits   # (batch, seq_len, vocab)

        # last REAL (non-pad) token position per sequence
        seq_lens = enc["attention_mask"].sum(dim=1) - 1
        last_logits = logits[torch.arange(len(batch)), seq_lens]   # (batch, vocab)

        for row_logits in last_logits:
            scores = {}
            for label, ids in label_token_ids.items():
                scores[label] = torch.logsumexp(row_logits[ids], dim=0).item()
            all_preds.append(max(scores, key=scores.get))
    return all_preds


def evaluate_predictions(y_true_labels, y_pred_labels, run_name):
    y_true = [LABEL2ID[l] for l in y_true_labels]
    y_pred = [LABEL2ID[l] for l in y_pred_labels]
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    precision, recall, f1_per_class, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    result = {
        "run_name": run_name, "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted,
        "per_class": {ID2LABEL[i]: {"precision": precision[i], "recall": recall[i], "f1": f1_per_class[i]} for i in range(3)},
        "confusion_matrix": cm.tolist(),
    }
    print(f"{run_name}: accuracy={acc:.4f} f1_macro={f1_macro:.4f} f1_weighted={f1_weighted:.4f}")
    return result


zero_shot_results = {}
skipped_candidates = []

ZERO_SHOT_BATCH_SIZE = 8   # conservative -- some candidates here run with ~500MB of headroom

for name, candidate in ZERO_SHOT_CANDIDATES.items():
    hf_id, quantize = candidate["hf_id"], candidate["quantize"]
    print(f"\n{'='*70}\nZero-shot: {name}{' (4-bit)' if quantize else ''}\n{'='*70}")

    # Defensive: clear any fragmentation left over from a previous candidate before this one
    # even starts, not just after it finishes.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = None
    try:
        tok = AutoTokenizer.from_pretrained(hf_id)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = "left"   # so the last real token is always at the same relative position

        load_kwargs = {"dtype": torch.bfloat16, "device_map": ({"": 0} if torch.cuda.is_available() else None)}
        if quantize:
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
            )
        model = AutoModelForCausalLM.from_pretrained(hf_id, **load_kwargs)

        label_token_ids = get_label_token_ids(tok)
        preds = zero_shot_predict_batch(model, tok, test_df["text"].tolist(), label_token_ids, batch_size=ZERO_SHOT_BATCH_SIZE)
        result = evaluate_predictions(test_df["label"].tolist(), preds, f"{name}_zero_shot")
        zero_shot_results[name] = result

        with mlflow.start_run(run_name=f"{name}_zero_shot"):
            mlflow.log_params({"model": name, "method": "zero_shot", "quantized": quantize})
            mlflow.log_metrics({"test_accuracy": result["accuracy"], "test_f1_macro": result["f1_macro"],
                                 "test_f1_weighted": result["f1_weighted"]})

    except Exception as e:
        # Gated models (Llama) fail to LOAD without HF_TOKEN + accepted license. Large models
        # can OOM during the forward pass itself, not just loading -- this except covers both,
        # since previously only the load call was wrapped and an inference-time OOM crashed the
        # whole run uncaught (and left the dead model's memory stuck for every cell after it).
        print(f"SKIPPING {name}: {type(e).__name__}: {str(e)[:200]}")
        if "gated" in str(e).lower() or "access" in str(e).lower():
            print(f"  -> looks like a gated model. Accept the license at https://huggingface.co/{hf_id} "
                  f"and set HF_TOKEN, then re-run this cell if you want it included.")
        elif "out of memory" in str(e).lower():
            print(f"  -> CUDA OOM. If this keeps happening, lower ZERO_SHOT_BATCH_SIZE above, or set "
                  f"quantize=True for this candidate in ZERO_SHOT_CANDIDATES and re-run.")
        skipped_candidates.append(name)

    finally:
        # Runs whether the trial succeeded, was skipped, or raised -- a model is never left
        # dangling on the GPU going into the next candidate.
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if skipped_candidates:
    print(f"\nSkipped {len(skipped_candidates)} candidate(s): {skipped_candidates}")


2026-08-01 13:06:10,605 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:06:10,621 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-08-01 13:06:10,669 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:06:10,686 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer_config.json "HTTP/1.1 200 OK"



Zero-shot: Qwen2.5-0.5B-Instruct


2026-08-01 13:06:10,734 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-01 13:06:10,783 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-01 13:06:11,568 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct "HTTP/1.1 200 OK"
2026-08-01 13:06:11,628 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:06:11,646 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-08-01 13:06:11,707 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-08-01 13:06:12,493 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:06:12,509 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/generation_config.json "HTTP/1.1 200 OK"


...

SmolLM2-1.7B-Instruct_zero_shot: accuracy=0.4384 f1_macro=0.4311 f1_weighted=0.4300


2026-08-01 13:24:04,512 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
2026-08-01 13:24:04,559 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"



Zero-shot: Llama-3.2-1B-Instruct
SKIPPING Llama-3.2-1B-Instruct: OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
401 Client Error. (Request ID: Root=1-6a6df374-276cdf31096447fd11c3c32
  -> looks like a gated model. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct and set HF_TOKEN, then re-run this cell if you want it included.

Zero-shot: Llama-3.2-3B-Instruct


2026-08-01 13:24:05,345 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
2026-08-01 13:24:05,395 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"


SKIPPING Llama-3.2-3B-Instruct: OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct.
401 Client Error. (Request ID: Root=1-6a6df375-08bc3736424a0d5c3b2df77
  -> looks like a gated model. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and set HF_TOKEN, then re-run this cell if you want it included.

Skipped 4 candidate(s): ['Qwen2.5-7B-Instruct', 'Gemma-2-2B-it', 'Llama-3.2-1B-Instruct', 'Llama-3.2-3B-Instruct']


## Zero-shot leaderboard -- picking which models advance to fine-tuning

Ranked by test F1 (weighted). Only the top `TOP_K_FOR_FINETUNING` advance to the expensive
LoRA/QLoRA search below -- this is a proxy, not a guarantee (a model that zero-shots poorly
could in principle fine-tune better than one that zero-shots well), but it's the standard,
defensible way to keep a multi-model benchmark's compute bounded: screen cheap, invest
expensive compute only where the cheap screen says it's worth it.


In [27]:
zero_shot_leaderboard = pd.DataFrame([
    {"model": name, "accuracy": r["accuracy"], "f1_macro": r["f1_macro"], "f1_weighted": r["f1_weighted"]}
    for name, r in zero_shot_results.items()
]).sort_values("f1_weighted", ascending=False).reset_index(drop=True)

print(zero_shot_leaderboard.to_string(index=True))
zero_shot_leaderboard.to_csv(os.path.join(RESULTS_DIR, "zero_shot_leaderboard.csv"), index=False)

FINETUNE_CANDIDATES = {
    name: ZERO_SHOT_CANDIDATES[name]["hf_id"]
    for name in zero_shot_leaderboard["model"].head(TOP_K_FOR_FINETUNING).tolist()
}
print(f"\nAdvancing to LoRA/QLoRA fine-tuning: {list(FINETUNE_CANDIDATES.keys())}")


                   model  accuracy  f1_macro  f1_weighted
0    Qwen2.5-3B-Instruct  0.489496  0.471524     0.464739
1  Qwen2.5-1.5B-Instruct  0.480392  0.450831     0.448035
2  SmolLM2-1.7B-Instruct  0.438375  0.431144     0.429989
3  Qwen2.5-0.5B-Instruct  0.415966  0.411073     0.410640
4  Phi-3.5-mini-instruct  0.414566  0.328948     0.334301

Advancing to LoRA/QLoRA fine-tuning: ['Qwen2.5-3B-Instruct', 'Qwen2.5-1.5B-Instruct']


## Stage 2 — LoRA and QLoRA fine-tuning (top `TOP_K_FOR_FINETUNING` zero-shot models only)

Both use `AutoModelForSequenceClassification` on the same decoder backbones (adds a linear
classification head reading the last non-padding token's hidden state -- the standard way to
fine-tune a causal LM for classification, and directly analogous to what FinBERT+LoRA did on
an encoder). QLoRA is identical except the frozen base weights are loaded 4-bit NF4-quantized
(`BitsAndBytesConfig`) before LoRA adapters are attached -- same adapters, much smaller memory
footprint, at some numerical precision cost.

Both get the same small hyperparameter search (rank, alpha, dropout, learning rate) before the
best configuration trains to convergence with early stopping on validation F1.


In [28]:
def tokenize_split(tokenizer, split_df):
    enc = tokenizer(split_df["text"].tolist(), padding="max_length", truncation=True,
                     max_length=MAX_LENGTH, return_tensors="pt")
    labels = torch.tensor(split_df["label_id"].tolist(), dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)


TARGET_MODULE_SETS = {
    "attn_only": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "attn_and_mlp": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}


def build_classifier(hf_id, quantize: bool, config: dict):
    quant_kwargs = {}
    if quantize:
        quant_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
    base = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
        dtype=torch.bfloat16, device_map=({"": 0} if torch.cuda.is_available() else None), **quant_kwargs,
    )
    base.config.pad_token_id = base.config.eos_token_id

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["r"], lora_alpha=config["alpha"], lora_dropout=config["dropout"],
        target_modules=TARGET_MODULE_SETS[config["target_modules"]], bias="none",
    )
    model = get_peft_model(base, peft_config)
    model.print_trainable_parameters()
    return model


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    loss_fn = nn.CrossEntropyLoss()

    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(model.device), attention_mask.to(model.device), labels.to(model.device)
        with torch.set_grad_enabled(training):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_fn(logits, labels)
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1


def train_with_early_stopping(model, train_loader, val_loader, lr, weight_decay, max_epochs, patience,
                               optuna_trial=None):
    """optuna_trial, if passed, gets an intermediate report()+prune check after every epoch --
    this is what lets Optuna kill a clearly-losing config after 1-2 epochs instead of wasting
    the full SEARCH_EPOCHS budget on it."""
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    best_val_f1, best_state, epochs_no_improve = -1, None, 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_f1 = run_epoch(model, train_loader, optimizer)
        val_loss, val_f1 = run_epoch(model, val_loader)
        print(f"  epoch {epoch}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

        if optuna_trial is not None:
            optuna_trial.report(val_f1, epoch)
            if optuna_trial.should_prune():
                raise optuna.TrialPruned()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items() if "lora" in k or "classifier" in k or "score" in k}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
    return model, best_val_f1


### The search space (7 dimensions, TPE-sampled, median-pruned)

| Hyperparameter | Range | Why it's worth tuning |
|---|---|---|
| `r` (LoRA rank) | 4 - 64 (log) | Adapter capacity -- too low underfits, too high risks overfitting a 7.5k-row train set |
| `alpha` | 8 - 128 (log) | Scales the adapter's effective update magnitude relative to `r` |
| `dropout` | 0.0 - 0.3 | Regularization on the adapter path specifically |
| `learning_rate` | 1e-5 - 1e-3 (log) | The single highest-leverage knob for adapter fine-tuning |
| `weight_decay` | 1e-4 - 0.1 (log) | L2 regularization on the trainable (adapter + head) params |
| `batch_size` | {8, 16, 32} | Affects both gradient noise and throughput |
| `target_modules` | attention-only vs. attention+MLP | Whether the adapter touches only attention projections or MLP blocks too -- a real architectural choice, not just a training knob |

25 trials per (model, method) combination x 4 combinations (2 models x {LoRA, QLoRA}) = 100
trials total. `MedianPruner` stops a trial the moment its validation F1 falls behind the
median of previously completed trials at the same epoch, so most of the 100 never run to
completion -- this is what makes 25 real trials per combo affordable rather than nominal.


In [29]:
def make_objective(hf_id, quantize: bool, train_ds, val_ds, tag: str):
    def objective(trial):
        config = {
            "r": trial.suggest_int("r", 4, 64, log=True),
            "alpha": trial.suggest_int("alpha", 8, 128, log=True),
            "dropout": trial.suggest_float("dropout", 0.0, 0.3),
            "lr": trial.suggest_float("lr", 1e-5, 1e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-4, 0.1, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32]),
            "target_modules": trial.suggest_categorical("target_modules", list(TARGET_MODULE_SETS.keys())),
        }
        print(f"\n[{tag}] trial {trial.number}: {config}")

        train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

        # model construction (loading + attaching LoRA) is INSIDE the try too -- an OOM there
        # is just as real as one during training, and previously wasn't covered by the cleanup.
        model = None
        try:
            model = build_classifier(hf_id, quantize, config)
            _, val_f1 = train_with_early_stopping(
                model, train_loader, val_loader, config["lr"], config["weight_decay"],
                SEARCH_EPOCHS, patience=99, optuna_trial=trial,
            )
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return val_f1
    return objective


def hyperparameter_search(hf_id, train_ds, val_ds, quantize: bool, tag: str):
    study = optuna.create_study(
        direction="maximize", sampler=TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=tag,
    )
    # catch=(...) means a trial that genuinely crashes (e.g. CUDA OOM on a large batch_size
    # draw) is recorded as FAILED and the study moves on, instead of the whole search dying.
    study.optimize(
        make_objective(hf_id, quantize, train_ds, val_ds, tag), n_trials=SEARCH_TRIALS,
        catch=(RuntimeError,),
    )

    trials_df = study.trials_dataframe()
    trials_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_optuna_trials.csv"), index=False)
    n_pruned = (trials_df["state"] == "PRUNED").sum()
    n_complete = (trials_df["state"] == "COMPLETE").sum()
    n_failed = (trials_df["state"] == "FAIL").sum()
    print(f"\n[{tag}] search done: {n_complete} completed, {n_pruned} pruned, {n_failed} failed (of {SEARCH_TRIALS})")
    if n_complete == 0:
        raise RuntimeError(
            f"[{tag}] every trial failed or was pruned -- likely every config OOM'd (check "
            f"{tag}_optuna_trials.csv for the actual errors). Try a smaller batch_size range "
            f"in the search space, or set quantize=True for this model if it isn't already."
        )
    print(f"[{tag}] best config: {study.best_params} (val_f1={study.best_value:.4f})")

    try:
        importances = optuna.importance.get_param_importances(study)
        print(f"[{tag}] parameter importances: {importances}")
    except Exception as e:
        print(f"[{tag}] importance computation skipped: {e}")

    best_config = study.best_params
    train_loader = DataLoader(train_ds, batch_size=best_config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=best_config["batch_size"], shuffle=False)
    print(f"\n[{tag}] training best config to full convergence ({FULL_EPOCHS} epochs, patience={PATIENCE})...")
    final_model = build_classifier(hf_id, quantize, best_config)
    final_model, final_val_f1 = train_with_early_stopping(
        final_model, train_loader, val_loader, best_config["lr"], best_config["weight_decay"], FULL_EPOCHS, PATIENCE,
    )
    return final_model, best_config, final_val_f1, study


In [ ]:
finetune_results = {}
finetune_models = {}

EVAL_BATCH_SIZE = 32   # only used for the held-out test set -- training batch size is tuned per trial

for name, hf_id in FINETUNE_CANDIDATES.items():
    tok = AutoTokenizer.from_pretrained(hf_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    train_ds = tokenize_split(tok, train_df)
    val_ds = tokenize_split(tok, val_df)
    test_ds = tokenize_split(tok, test_df)

    for method, quantize in [("lora", False), ("qlora", True)]:
        tag = f"{name}_{method}"
        print(f"\n{'#'*70}\n{tag}\n{'#'*70}")
        model, best_config, val_f1, study = hyperparameter_search(hf_id, train_ds, val_ds, quantize, tag)

        model.eval()
        test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
        all_preds, all_labels = [], []
        with torch.no_grad():
            for input_ids, attention_mask, labels in test_loader:
                logits = model(input_ids=input_ids.to(model.device), attention_mask=attention_mask.to(model.device)).logits
                all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
                all_labels.extend(labels.tolist())

        pred_labels = [ID2LABEL[p] for p in all_preds]
        true_labels = [ID2LABEL[l] for l in all_labels]
        result = evaluate_predictions(true_labels, pred_labels, tag)
        result["best_config"] = {k: (int(v) if isinstance(v, (np.integer, int)) else float(v)) if not isinstance(v, str) else v
                                  for k, v in best_config.items()}
        result["n_trials_run"] = len(study.trials)
        result["n_trials_pruned"] = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        finetune_results[tag] = result
        finetune_models[tag] = model

        adapter_path = os.path.join(MODELS_DIR, f"{tag}_adapter")
        model.save_pretrained(adapter_path)
        tok.save_pretrained(os.path.join(MODELS_DIR, f"{tag}_tokenizer"))

        with mlflow.start_run(run_name=tag):
            mlflow.log_params({"model": name, "method": method, **result["best_config"]})
            mlflow.log_metrics({"val_f1": val_f1, "test_accuracy": result["accuracy"],
                                 "test_f1_macro": result["f1_macro"], "test_f1_weighted": result["f1_weighted"],
                                 "n_trials_run": result["n_trials_run"], "n_trials_pruned": result["n_trials_pruned"]})
            mlflow.log_artifacts(adapter_path, artifact_path="adapter")


2026-08-01 13:25:04,115 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:25:04,132 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json "HTTP/1.1 200 OK"
2026-08-01 13:25:04,182 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-01 13:25:04,199 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-01 13:25:04,252 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-3B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-01 13:25:04,306 | INFO | HTTP Request: GET https://huggingface


######################################################################
Qwen2.5-3B-Instruct_lora
######################################################################

[Qwen2.5-3B-Instruct_lora] trial 0: {'r': 10, 'alpha': 112, 'dropout': 0.21959818254342153, 'lr': 0.00015751320499779721, 'weight_decay': 0.00029380279387035364, 'batch_size': 32, 'target_modules': 'attn_and_mlp'}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-08-01 13:25:56,689 | WARNING | Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 18,714,624 || all params: 3,104,659,456 || trainable%: 0.6028


...

[W 2026-08-01 13:26:00,797] Trial 1 failed with parameters: {'r': 4, 'alpha': 118, 'dropout': 0.2497327922401265, 'lr': 2.6587543983272695e-05, 'weight_decay': 0.0003511356313970409, 'batch_size': 32, 'target_modules': 'attn_only'} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 594.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 138.81 MiB is free. Including non-PyTorch memory, this process has 14.42 GiB memory in use. Of the allocated memory 14.05 GiB is allocated by PyTorch, and 258.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)').
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
   


[Qwen2.5-3B-Instruct_lora] trial 2: {'r': 21, 'alpha': 11, 'dropout': 0.08764339456056544, 'lr': 5.4041038546473305e-05, 'weight_decay': 0.0023345864076016252, 'batch_size': 8, 'target_modules': 'attn_only'}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 9,682,944 || all params: 3,095,627,776 || trainable%: 0.3128


## Final comparison

Every zero-shot candidate that loaded successfully, plus LoRA and QLoRA results for whichever
`TOP_K_FOR_FINETUNING` models won that screen, plus FinBERT+LoRA's known 86.5% F1 for
reference -- all on the identical test split.


In [ ]:
comparison_rows = []
for name, result in zero_shot_results.items():
    comparison_rows.append({"model": name, "method": "zero_shot", "accuracy": result["accuracy"],
                             "f1_macro": result["f1_macro"], "f1_weighted": result["f1_weighted"]})
for tag, result in finetune_results.items():
    model_name, method = tag.rsplit("_", 1)
    comparison_rows.append({"model": model_name, "method": method, "accuracy": result["accuracy"],
                             "f1_macro": result["f1_macro"], "f1_weighted": result["f1_weighted"]})
comparison_rows.append({"model": "FinBERT+LoRA (Part A reference)", "method": "lora",
                         "accuracy": 0.8654, "f1_macro": None, "f1_weighted": 0.8652})

comparison_df = pd.DataFrame(comparison_rows).sort_values("f1_weighted", ascending=False)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(RESULTS_DIR, "llm_lora_qlora_comparison.csv"), index=False)

best_tag = max(finetune_results, key=lambda k: finetune_results[k]["f1_weighted"])
best_result = finetune_results[best_tag]
best_model = finetune_models[best_tag]
print(f"\nBest fine-tuned variant: {best_tag} (f1_weighted={best_result['f1_weighted']:.4f})")
print(f"Confusion matrix ({best_tag}):")
print(pd.DataFrame(best_result["confusion_matrix"], index=["true_pos", "true_neu", "true_neg"],
                    columns=["pred_pos", "pred_neu", "pred_neg"]))


## Sanity check — does this fix FinBERT's known blind spot?

FinBERT+LoRA was 99%+ confidently WRONG on "corporate action" headlines that don't contain
classic profit/loss vocabulary (verified live against the dashboard's FastAPI service):
partnership announcements, new labs, contract wins, expansions. Testing the same failing
headlines against the best model found here.


In [ ]:
BLIND_SPOT_HEADLINES = [
    ("TCS opens industrial AI lab in Bengaluru in partnership with Nvidia", "positive/neutral"),
    ("TCS partners with Nvidia", "positive/neutral"),
    ("Company launches new product line", "positive/neutral"),
    ("Company expands operations to new city", "positive/neutral"),
    ("Infosys wins new contract worth 500 crore", "positive"),
    ("Company invests in new manufacturing facility", "positive/neutral"),
    ("Reliance Industries posts record quarterly profit", "positive"),   # control: should stay positive
    ("Stock rallies after strong earnings", "positive"),                  # control
]

best_tok = AutoTokenizer.from_pretrained(os.path.join(MODELS_DIR, f"{best_tag}_tokenizer"))
best_model.eval()
print(f"{'Headline':<70} {'Expected':<18} {'Predicted'}")
print("-" * 105)
with torch.no_grad():
    for headline, expected in BLIND_SPOT_HEADLINES:
        enc = best_tok(headline, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(best_model.device)
        logits = best_model(**enc).logits
        pred = ID2LABEL[logits.argmax(dim=-1).item()]
        probs = torch.softmax(logits, dim=-1)[0]
        conf = probs.max().item()
        print(f"{headline[:68]:<70} {expected:<18} {pred} ({conf:.1%} confident)")


In [ ]:
# ══════════════════════════════════════════════════════════════
# SAVE FINAL SUMMARY
# ══════════════════════════════════════════════════════════════
summary = {
    "comparison": comparison_rows,
    "best_variant": best_tag,
    "best_metrics": {k: v for k, v in best_result.items() if k != "confusion_matrix"},
}
with open(os.path.join(RESULTS_DIR, "llm_lora_qlora_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSaved comparison table + summary to {RESULTS_DIR}/")
print(f"Best adapter saved to {MODELS_DIR}/{best_tag}_adapter/")
print("\nTo serve the fine-tuned model locally via Ollama afterward: merge the LoRA adapter into")
print("the base weights (peft's `merge_and_unload()`), export to GGUF with llama.cpp's converter,")
print("then `ollama create <name> -f Modelfile`.")
